# Topic: LAG and LEAD

## Definition (30-second explanation)
* `LAG()` and `LEAD()` are window functions that allow you to access values from previous (LAG) or subsequent (LEAD) rows within a result set, relative to the current row.
* They allow for cross-row comparisons (like calculating period-over-period growth) without requiring computationally expensive self-joins.

## Why Interviewers Ask This
* To test your ability to perform time-series analysis and calculate standard business metrics (MoM, YoY growth).
* To see if you can write clean, performant SQL (using window functions instead of nested subqueries/joins).
* To verify you know how to handle edge cases like `NULL` values on the first/last rows and division-by-zero errors.

## Core Concepts
* **Offset:** The second parameter in the function (e.g., `LAG(revenue, 1)`) determines how many rows to look back/forward. The default is 1.
* **Default Value:** The third parameter (e.g., `LAG(revenue, 1, 0)`) defines what to return if the target row doesn't exist (e.g., the very first row), preventing unexpected `NULL`s.
* **PARTITION BY:** Critical for resetting the window per group (e.g., looking at previous revenue *for the same product*, not just the absolute previous row).

## When to Use
* **LAG:** Calculating day-over-day or month-over-month changes, or finding the time elapsed between consecutive user events (sessionization).
* **LEAD:** Looking forward to predict next states, checking if a user upgraded in their next billing cycle, or identifying trend reversals.

## Advantages
* Dramatically simplifies SQL logic compared to self-joins.
* Highly performant since the database engine only needs to scan the data partition once.
* CTEs combined with LAG/LEAD make complex trend logic highly readable.

## Limitations
* Requires a strict, deterministic `ORDER BY` inside the `OVER()` clause to ensure adjacent rows are chronologically correct.
* The output is sensitive to missing data (e.g., if a month is missing from the table, `LAG(..., 1)` will grab the month before that, skipping a month).

## Common Comparisons
* **LAG vs LEAD:** `LAG` looks backwards (historical comparison), `LEAD` looks forwards (future outcome tracking). 
* **LAG vs Self-Join:** LAG is linear and clean; Self-Joins require joining a table on `t1.date = t2.date + 1`, which is slow and prone to duplication if relationships aren't 1:1.

## Common Interview Traps
* **Missing PARTITION BY:** If omitted, `LAG` might compare the January revenue of a 'Phone' to the December revenue of a 'Laptop' just because they are adjacent in the table.
* **Division by Zero:** When calculating growth percentage `(current - prev) / prev`, if `prev` is 0, the query crashes. Always use `NULLIF(prev, 0)`.
* **Cluttered SELECT clauses:** Writing the full `LAG(...)` window function multiple times in the same `SELECT` (once for absolute change, once for percentage). Use a CTE to define it once.

## Python / SQL Syntax
```sql
-- Standard syntax with offset (1) and default value (0)
LAG(column_name, 1, 0) OVER (
    PARTITION BY group_col 
    ORDER BY sort_col
)
```

## Important Formula
**Percentage Change:** (current_value - previous_value) / NULLIF(previous_value, 0)

## 45-Second Interview Answer
"LAG and LEAD are window functions used to access data from adjacent rows without needing self-joins. LAG looks backward, which is perfect for period-over-period growth or time-between-events, while LEAD looks forward. In interviews, the key to using them correctly is ensuring you have a strict ORDER BY for chronological alignment, a proper PARTITION BY so you don't accidentally compare different categories, and handling edge cases—like providing a default value for the first row to prevent NULLs, and using NULLIF to prevent division-by-zero when calculating growth rates."

## Example Questions:

### Q1. Calculate year-over-year revenue change for each product.
* **Ideal Interview Answer:** I would use `LAG(revenue, 1)` over a partition of the `product` ordered by `year`. To calculate the percentage change, I'd subtract the lagged revenue from the current revenue, and divide by `NULLIF(lagged_revenue, 0)` to prevent division by zero errors on products that had 0 revenue in the previous year.
* **Common Mistakes:** Using an offset of 12 on daily/monthly data without explicitly aggregating it to the yearly level first, which breaks if a month/day is missing.
* **Likely Follow-up:** "How would you write this if you wanted Month-over-Month growth, but some products have missing months in the dataset?"

### Q2. Find all months where revenue dropped more than 10% from the previous month.
* **Ideal Interview Answer:** First, I'd use a CTE to create a `prev_month_revenue` column using `LAG()`. In the main query, I would calculate the ratio `revenue / prev_month_revenue`. I would filter in the `WHERE` clause for `revenue < prev_month_revenue * 0.90` (or where the calculated percentage drop is < -0.10). 
* **Common Mistakes:** Trying to put the `LAG` function directly inside the `WHERE` clause, which SQL does not allow.
* **Likely Follow-up:** "Why use a CTE instead of writing the LAG function twice in the SELECT and WHERE clauses?"

### Q3. Calculate a 3-month moving average using LAG.
* **Ideal Interview Answer:** While you *could* do `(revenue + LAG(revenue, 1) + LAG(revenue, 2)) / 3`, the much better and dynamic way to do this in SQL is using the window frame clause: `AVG(revenue) OVER (PARTITION BY product ORDER BY month ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)`. 
* **Common Mistakes:** Hardcoding `LAG(..., 1)` and `LAG(..., 2)` which becomes unscalable if the requirement changes to a 12-month moving average.
* **Likely Follow-up:** "What happens to the moving average for the very first month of a product's lifecycle using the `ROWS BETWEEN` method?"

### Q4. Detect revenue trend reversals (went up then down, or down then up).
* **Ideal Interview Answer:** I would use a CTE with both `LAG()` and `LEAD()`. I'd compare the current month's revenue to both. A "peak" reversal is when `revenue > LAG(revenue)` AND `revenue > LEAD(revenue)`. A "valley" reversal is when `revenue < LAG(revenue)` AND `revenue < LEAD(revenue)`.
* **Common Mistakes:** Overcomplicating it with multiple nested CTEs instead of grabbing both the prior and next values in a single CTE pass.
* **Likely Follow-up:** "How would you handle a scenario where the revenue stayed exactly flat for a month before dropping?"

### Q5. Find the product with the most consecutive months of growth.
* **Ideal Interview Answer:** This is a classic "gaps and islands" problem. First, I'd use `LAG` to flag if a month grew compared to the previous (1 for growth, 0 for no growth). Then, I'd use a cumulative sum (`SUM() OVER()`) of the *non-growth* flags to group consecutive growth months into unique "islands". Finally, I'd count the size of each island, rank them, and select the maximum.
* **Common Mistakes:** Thinking this can be solved with a single `LAG` statement without recognizing it requires iterative grouping logic (gaps and islands).
* **Likely Follow-up:** "Can you write the pseudo-code for the 'islands' grouping step?"

## Practice Questions:

### Q1:
**Scenario:**
You are a Data Scientist analyzing customer retention. The marketing team wants to know the number of days that elapsed between a user's current purchase and their immediately preceding purchase. If it is the user's very first purchase, the "days since last purchase" should be 0.

**Mock Schema:**
```sql
-- DDL
CREATE TABLE purchases (
    purchase_id INT AUTO_INCREMENT PRIMARY KEY,
    user_id INT,
    purchase_date DATE,
    amount DECIMAL(10, 2)
);

-- Mock Data
INSERT INTO purchases (user_id, purchase_date, amount) VALUES
(101, '2026-08-01', 50.00),
(101, '2026-08-05', 75.50),  -- 4 days since last
(101, '2026-08-15', 20.00),  -- 10 days since last
(102, '2026-08-02', 100.00), -- 0 days since last
(102, '2026-08-03', 45.00),  -- 1 day since last
(103, '2026-08-10', 60.00);  -- 0 days since last
```

* **Answer:** 
```sql
WITH prev_purchases AS (
    SELECT 
        user_id, 
        purchase_date,
        LAG(purchase_date) OVER(PARTITION BY user_id ORDER BY purchase_date) as prev_purchase
    FROM purchases
)
SELECT 
    user_id, 
    purchase_date,
    COALESCE(DATEDIFF(purchase_date, prev_purchase), 0) AS days_since_last_purchase
FROM prev_purchases;
```
* **Common Mistakes:** 
    * Using a `GROUP BY` clause, which collapses the transaction-level data and destroys the intermediate purchase history.
    * Forgetting to handle the `NULL` value on the user's very first row, which results in a `NULL` calculation instead of the requested `0`.
* **Interview Tip:** Whenever calculating "time since last [event]" (often called sessionization), `LAG` is your best friend. Wrap it in a CTE, and simply run a date difference function on the outer query. Never use `GROUP BY` unless you are actively trying to reduce the number of rows (e.g., finding only the maximum purchase).